# Train and evaluate the project DQN on Google Colab

This notebook runs the repository's PyTorch DQN with the linked project's Atari data-preparation pipeline. Select a GPU runtime for practical training; the code automatically uses CUDA when available. The default experiment is still a compact 250,000-decision run, not an exact reproduction of the full linked training protocol. Colab can disconnect; periodic model checkpoints are written to Drive, but these are inference checkpoints and do not resume optimizer/replay state.

In Colab, select **Runtime → Change runtime type → T4 GPU** (or another available GPU). Run cells from top to bottom. The Atari ROM distributed with ALE is subject to its own license; use it only where permitted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

PROJECT_DIR = Path('/content/DQN')
if not (PROJECT_DIR / 'pyproject.toml').exists():
    !git clone https://github.com/bhagathchan1/DQN.git {PROJECT_DIR}
os.chdir(PROJECT_DIR)
print('Project:', PROJECT_DIR)

In [ ]:
%pip install -e '.[env]'
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Choose Runtime → Change runtime type → T4 GPU, then rerun.'

In [ ]:
# Check Gymnasium + ALE + Breakout before starting a long run.
%run scripts/environment_smoke_test.py --pipeline upstream

## Prepare a reviewable transition sample

The linked DQN project generates its replay data online. This optional cell collects a small random-policy sample with the upstream-style Atari preprocessing pipeline and saves the transition arrays plus a four-frame visualization to Drive. It is a pipeline inspection artifact, not the agent's learned replay buffer.

In [ ]:
from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/DQN')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
sample_path = DRIVE_DIR / 'data/breakout_transition_sample.npz'
!python scripts/collect_transition_sample.py --steps 256 --seed 7 --output {sample_path}
print('Sample archive:', sample_path)
print('Preview:', sample_path.with_name(sample_path.stem + '_first_state.png'))

In [ ]:
# Keep durable files on Drive. Update RUN_ID to create a separate experiment.
from pathlib import Path
import shutil, yaml

DRIVE_DIR = Path('/content/drive/MyDrive/DQN')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
for name in ('checkpoints', 'results'):
    local = PROJECT_DIR / name
    durable = DRIVE_DIR / name
    durable.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        local.unlink()
    elif local.exists():
        shutil.move(str(local), str(durable / 'initial_repository_artifacts'))
    local.symlink_to(durable, target_is_directory=True)

RUN_ID = 'breakout_colab_upstream_data_seed7'
config = yaml.safe_load((PROJECT_DIR / 'configs/default.yaml').read_text())
config['run']['id'] = RUN_ID
config['environment']['pipeline'] = 'upstream'
config['training']['device'] = 'auto'
# Adjust this before training if your Colab session has a shorter time budget.
config['training']['total_environment_steps'] = 250_000
config['training']['checkpoint_frequency'] = 25_000
config_path = PROJECT_DIR / 'configs/colab_upstream_data.yaml'
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print('Config:', config_path)
print('Checkpoints and metrics will persist in:', DRIVE_DIR)

In [ ]:
# Train. CUDA is used automatically; every 25,000 decisions saves model weights to Google Drive.
!dqn-train --config configs/colab_upstream_data.yaml

In [ ]:
# Evaluate the final checkpoint and save the episode-level results on Drive.
checkpoint = DRIVE_DIR / 'checkpoints' / f'{RUN_ID}_final.pt'
evaluation = DRIVE_DIR / 'results/metrics' / f'{RUN_ID}_evaluation.json'
!dqn-evaluate --config configs/colab_upstream_data.yaml --checkpoint {checkpoint} --output {evaluation}
print('Evaluation:', evaluation)

## Inspect the run

Training episode rewards are in `MyDrive/DQN/results/metrics/`. The final weights are in `MyDrive/DQN/checkpoints/`. To compare against the paper, see `docs/upstream_comparison.md`; this run uses the upstream-style data pipeline but keeps the project's compact training budget and learner settings. The GIFs and figures under `assets/upstream_dqn/` are original upstream reference assets, not outputs from this training run.